 **Abstractive Text Summarization using BART**

*   BART - Bidirectional and Auto Regressie Transformer(BERT + GPT)



In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import pandas as pd

In [ ]:
!pip install "transformers<5.0.0"

In [ ]:
!pip install rouge-score bert-score nltk
import nltk
nltk.download('punkt')

In [ ]:
!pip install datasets

In [ ]:
#Loading the dataset
from datasets import load_dataset
ds = load_dataset("knkarthick/dialogsum")

In [ ]:
ds

In [ ]:
ds['train'][1]['dialogue']

In [ ]:
ds['train'][1]['summary']

1. WITHOUT FINE - TUNING


In [ ]:
from transformers import pipeline

pipe = pipeline("summarization", model="facebook/bart-large-cnn")

In [ ]:
article_1 = ds['train'][1]['dialogue']

In [ ]:
pipe(article_1, max_length= 20, min_length= 10, do_sample= False)




2.   With Fine Tuning



In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
#Tokenization
def preprocess_function(batch):
    source = batch['dialogue']
    target = batch['summary']
    source_ids = tokenizer(source, truncation= True, padding= 'max_length', max_length= 128)
    target_ids = tokenizer(target, truncation= True, padding= 'max_length', max_length= 128)

    #labels for loss computation
    labels = target_ids['input_ids']
    labels = [[(label if label != tokenizer.pad_token_id else -100 )
                for label in labels_example]
                for labels_example in labels]

    return{
        "input_ids": source_ids['input_ids'],
        "attention_mask": source_ids['attention_mask'],
        "labels": labels
    }

In [ ]:
df_source = ds.map(preprocess_function, batched= True )

In [ ]:
#training arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir= 'bart_model',
    per_device_train_batch_size= 8,
    num_train_epochs = 2,
    remove_unused_columns= True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = df_source['train'], eval_dataset = df_source['test']
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()

In [ ]:
eval_results

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
model.generation_config.early_stopping = True
model.generation_config.num_beams = 4
model.generation_config.length_penalty = 2.0

#Saving The Model
model.save_pretrained('/content/drive/MyDrive/bart_nlp')
tokenizer.save_pretrained('/content/drive/MyDrive/bart_nlp')

tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/bart_nlp')
model = AutoModelForSeq2SeqLM.from_pretrained('/content/drive/MyDrive/bart_nlp')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def summarize(blog_post):
    inputs = tokenizer(
        blog_post,
        max_length=1024,
        truncation=True,
        return_tensors='pt'
    ).to(device)

    summary_ids = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=150,
        min_length=10,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
        forced_bos_token_id=tokenizer.bos_token_id
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

#evaluate_summary() function
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def evaluate_summary(generated_summary, reference_summary):
    # BLEU
    reference_tokens = [reference_summary.split()]
    generated_tokens = generated_summary.split()
    smoothie = SmoothingFunction().method4
    bleu = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smoothie)

    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(reference_summary, generated_summary)

    # BERTScore
    P, R, F1 = bert_score([generated_summary], [reference_summary], lang='en', verbose=False)

    print("=" * 50)
    print(f"BLEU Score       : {bleu:.4f}")
    print("-" * 50)
    print(f"ROUGE-1  → P: {rouge_scores['rouge1'].precision:.4f} | R: {rouge_scores['rouge1'].recall:.4f} | F1: {rouge_scores['rouge1'].fmeasure:.4f}")
    print(f"ROUGE-2  → P: {rouge_scores['rouge2'].precision:.4f} | R: {rouge_scores['rouge2'].recall:.4f} | F1: {rouge_scores['rouge2'].fmeasure:.4f}")
    print(f"ROUGE-L  → P: {rouge_scores['rougeL'].precision:.4f} | R: {rouge_scores['rougeL'].recall:.4f} | F1: {rouge_scores['rougeL'].fmeasure:.4f}")
    print("-" * 50)
    print(f"BERTScore → P: {P.mean():.4f} | R: {R.mean():.4f} | F1: {F1.mean():.4f}")
    print("=" * 50)

    return {
        "bleu": bleu,
        "rouge1": rouge_scores['rouge1'],
        "rouge2": rouge_scores['rouge2'],
        "rougeL": rouge_scores['rougeL'],
        "bert_F1": F1.mean().item()
    }

#WITHOUT Fine-Tuning evaluation
article_1 = ds['train'][1]['dialogue']
reference_1 = ds['train'][1]['summary']

no_ft_summary = pipe(article_1, max_length=20, min_length=10, do_sample=False)[0]['summary_text']

print("\n=== WITHOUT Fine-Tuning ===")
print(f"Generated : {no_ft_summary}")
print(f"Reference : {reference_1}")
metrics_no_ft = evaluate_summary(no_ft_summary, reference_1)

#WITH Fine-Tuning evaluation
ft_summary = summarize(article_1)

print(f"\n[DEBUG] Fine-tuned raw output: '{ft_summary}'")
if not ft_summary.strip():
    print("[WARNING] Model generated empty output. Check device placement and generation params.")

print("\n=== WITH Fine-Tuning ===")
print(f"Generated : {ft_summary}")
print(f"Reference : {reference_1}")
if ft_summary.strip():
    metrics_ft = evaluate_summary(ft_summary, reference_1)
else:
    print("Skipping evaluation — generated summary is empty.")

# ── Side-by-side Comparison
comparison = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "BERTScore F1"],
    "Without Fine-Tuning": [
        metrics_no_ft["bleu"],
        metrics_no_ft["rouge1"].fmeasure,
        metrics_no_ft["rouge2"].fmeasure,
        metrics_no_ft["rougeL"].fmeasure,
        metrics_no_ft["bert_F1"]
    ],
    "With Fine-Tuning": [
        metrics_ft["bleu"],
        metrics_ft["rouge1"].fmeasure,
        metrics_ft["rouge2"].fmeasure,
        metrics_ft["rougeL"].fmeasure,
        metrics_ft["bert_F1"]
    ]
})

comparison["Improvement"] = (comparison["With Fine-Tuning"] - comparison["Without Fine-Tuning"]).round(4)
print("\n", comparison.to_string(index=False))